### Soma total de vendas por dia, ordens com status de 'Delivered' e State = NY

In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
# Criando várias tabelas temporárias 
silver_map = {
    # 'tmp_silver_customer': f'{silver_path}/customer/',
    'tmp_silver_orders': f'{silver_path}/orders/',
    # 'tmp_silver_product': f'{silver_path}/product/'
}
for view_name, path in silver_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
%python
# Salvando no formato delta na camada Gold
df_sales_ny_Gold = spark.sql("""
SELECT
shipped_date,
ROUND(SUM(total_sale), 2) AS total_sale
-- state,
-- status
FROM tmp_silver_orders
WHERE state = 'NY'
AND status = 'Delivered'
AND shipped_date IS NOT NULL
GROUP BY shipped_date
-- GROUP BY state, status
""")

# Salvar e transformar em formato delta na camada Gold
df_sales_ny_Gold.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{gold_path}/sales_ny')

In [0]:
%sql
/* Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)
CREATE TABLE IF NOT EXISTS bikestore.logistics.gold_sales_ny
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/sales_ny';*/

In [0]:
-- SELECT * FROM bikestore.logistics.gold_sales_ny;